# Functional Enrichment

Analysis of peaks for gene set enrichment with rGREAT.

### Build Gene Database

In [1]:
# Initialize
library(rtracklayer)
library(GenomicRanges)
library(rGREAT)

# Move to working directory
setwd("/home/dalbao/AlbaoRunx3Manuscript/cutnrun")

gtf <- import("source_data/genes.gtf")
tx  <- gtf[gtf$type == "transcript"]

# --- Filter to genes that can plausibly appear in your gene sets ---
keep_bt <- c("protein_coding", "lncRNA")     # add "TR_*"/"IG_*" if relevant to you
tx <- tx[tx$gene_biotype %in% keep_bt]

# Optional: drop poorly-supported isoforms so a spurious transcript
# doesn't drag the TSS upstream. TSL 1-3 is a common cut.
# tx <- tx[tx$transcript_support_level %in% c("1","2","3","NA")]

tx <- tx[!is.na(tx$gene_name) & tx$gene_name != ""]

# --- Collapse to one most-5' TSS per gene (strand-aware) ---
spans <- range(split(tx, tx$gene_name))
spans <- spans[elementNROWS(spans) == 1]     # drop multi-strand/scaffold artifacts
gene_span <- unlist(spans)

tss <- resize(gene_span, width = 1, fix = "start")
mcols(tss) <- NULL
tss$gene_id <- names(gene_span)              # SYMBOL, to match gene sets
names(tss) <- NULL
tss <- tss[!is.na(tss$gene_id) & tss$gene_id != ""]
tss <- tss[!duplicated(tss$gene_id)]

length(tss)     # expect ~20-35k with the filter above, not ~55k

Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomeInfoDb



[1] 21913

In [2]:
head(tss)

GRanges object with 6 ranges and 1 metadata column:
      seqnames    ranges strand |       gene_id
         <Rle> <IRanges>  <Rle> |   <character>
  [1]       12  85824550      - | 0610007P14Rik
  [2]       11  51688874      - | 0610009B22Rik
  [3]       11 120348678      + | 0610009L18Rik
  [4]       18  38250249      + | 0610009O20Rik
  [5]       11  23633639      - | 0610010F05Rik
  [6]       11  70237914      - | 0610010K14Rik
  -------
  seqinfo: 73 sequences from an unspecified genome; no seqlengths

### Peak Import and Chromosome End Attachment

In [3]:
# Load Peaks
peaks.all <- import("01_peakEDA/all.clean.noCluster.bed")
peaks.cl1 <- import("01_peakEDA/cluster1.clean.noCluster.bed")
peaks.cl2 <- import("01_peakEDA/cluster2.clean.noCluster.bed")

# Restrict to primary chromosomes; scaffolds add noise and no power
main <- c(1:19, "X")   # mm10/mm39
tss   <- keepSeqlevels(tss, main, pruning.mode = "coarse")
peaks.all <- keepSeqlevels(peaks.all, main, pruning.mode = "coarse")
peaks.cl1 <- keepSeqlevels(peaks.cl1, main, pruning.mode = "coarse")
peaks.cl2 <- keepSeqlevels(peaks.cl2, main, pruning.mode = "coarse")

# Attach chromosome lengths to TSS GRanges object, so GREAT can work
fai <- read.table("source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/genome.fa.fai")[, 1:2]
sl  <- setNames(fai$V2, fai$V1)
seqlengths(tss) <- sl[seqlevels(tss)]

### Process Gene Sets

In [4]:
# Load gene sets
gene.sets <- read.csv("/home/dalbao/CommonGeneSets/ConvertedLists/v11/genesets_v11_Ensembl98.csv", header = TRUE, stringsAsFactors = FALSE)
head(gene.sets)

,gs_name,gene_symbol,EnsemblID
,<chr>,<chr>,<chr>
1,TXM_lib-8C,Abi2,ENSMUSG00000026782
2,TXM_lib-8C,Acsf3,ENSMUSG00000015016
3,TXM_lib-8C,Adam8,ENSMUSG00000025473
4,TXM_lib-8C,Ano10,ENSMUSG00000037949
5,TXM_lib-8C,Cables1,ENSMUSG00000040957
6,TXM_lib-8C,Cd9,ENSMUSG00000030342


In [5]:
# Keep only gene_symbol if it exists in the TSS object
gene.sets <- gene.sets[gene.sets$gene_symbol %in% tss$gene_id, ]

# Process gene sets into a list of unique gene symbols per gene set name
gene.sets <- split(gene.sets$gene_symbol, gene.sets$gs_name)
gene.sets <- lapply(gene.sets, unique)

In [6]:
tss$gene_id[tss$gene_id %in% c("Sept6", "Septin6")]  # check that Runx3 is present in the TSS object
gene.sets$gs_name[grep("Sept", gene.sets$gs_name)]  # check that Runx3 is present in the gene sets``

[1] "Sept6"

NULL

In [7]:
res.all <- great(
    gr                = peaks.all,
    gene_sets         = gene.sets,
    tss_source        = tss,
    mode              = "basalPlusExt",   # GREAT default
    basal_upstream    = 5000,
    basal_downstream  = 1000,
    extension         = 1000000,
    min_gene_set_size = 5,
    exclude           = NULL,             # "gap" needs a recognized genome
    cores             = 4
)

res.cl1 <- great(
    gr                = peaks.cl1,
    gene_sets         = gene.sets,
    tss_source        = tss,
    mode              = "basalPlusExt",   # GREAT default
    basal_upstream    = 5000,
    basal_downstream  = 1000,
    extension         = 1000000,
    min_gene_set_size = 5,
    exclude           = NULL,             # "gap" needs a recognized genome
    cores             = 4
)

res.cl2 <- great(
    gr                = peaks.cl2,
    gene_sets         = gene.sets,
    tss_source        = tss,
    mode              = "basalPlusExt",   # GREAT default
    basal_upstream    = 5000,
    basal_downstream  = 1000,
    extension         = 1000000,
    min_gene_set_size = 5,
    exclude           = NULL,             # "gap" needs a recognized genome
    cores             = 4
)

* TSS extension mode is 'basalPlusExt'.

* construct the basal domains by extending 5000bp to upstream and 1000bp to downsteram of TSS.

* calculate distances to neighbour regions.

* extend to both sides until reaching the neighbour genes or to the maximal extension.

* check gene ID type in `gene_sets` and in `extended_tss`.

* use whole genome as background.

* overlap `gr` to background regions (based on midpoint).

* in total 4172 `gr`.

* overlap extended TSS to background regions.

* check which genes are in the gene sets.

* only take gene sets with size >= 5.

* in total 186 gene sets.

* overlap `gr` to every extended TSS.

* perform binomial test for each biological term.

* TSS extension mode is 'basalPlusExt'.

* construct the basal domains by extending 5000bp to upstream and 1000bp to downsteram of TSS.

* calculate distances to neighbour regions.

* extend to both sides until reaching the neighbour genes or to the maximal extension.

* check gene ID type in `gene_sets` a

In [8]:
tb.all <- getEnrichmentTable(res.all)
tb.cl1 <- getEnrichmentTable(res.cl1)
tb.cl2 <- getEnrichmentTable(res.cl2)

In [9]:
# Make folder if it does not exist
if (!dir.exists("05_enrichment")) {
    dir.create("05_enrichment", recursive = TRUE)
}

saveData <- function(df, filename){
    write.csv(df, paste0("05_enrichment/", filename))
}

saveData(tb.all , "all.peaks.enrichment.csv")
saveData(tb.cl1 , "cl1.peaks.enrichment.csv")
saveData(tb.cl2 , "cl2.peaks.enrichment.csv")

In [12]:
assoc_all <- getRegionGeneAssociations(res.all)
assoc_cl1 <- getRegionGeneAssociations(res.cl1)
assoc_cl2 <- getRegionGeneAssociations(res.cl2)

In [13]:
head(assoc_all)

GRanges object with 6 ranges and 3 metadata columns:
      seqnames            ranges strand |             name annotated_genes
         <Rle>         <IRanges>  <Rle> |      <character> <CharacterList>
  [1]        1   7397451-7398450      * |  shRunx3_Runx1_5          Pcmtd1
  [2]        1   9847901-9848700      * |  shCd19_Runx1_27     Sgk3,Mcmdc2
  [3]        1 10037651-10038650      * |   shCd19_Runx3_1     Cspp1,Cops5
  [4]        1 13371951-13373200      * |   shCd19_Runx3_2    Prdm14,Ncoa2
  [5]        1 13374001-13374750      * |  shCd19_Runx1_46           Ncoa2
  [6]        1 13382101-13382950      * | shRunx3_Runx3_33     Ncoa2,Tram1
        dist_to_TSS
      <IntegerList>
  [1]        308531
  [2]  49794,-59938
  [3]           0,0
  [4]   -244788,883
  [5]             0
  [6]  -8018,206960
  -------
  seqinfo: 20 sequences from an unspecified genome; no seqlengths